In [2]:
from bs4 import BeautifulSoup # HTML 다루는 기능
import requests
import pandas as pd

In [6]:


def make_data(html, brand, title, discount, price, etc, star, starlen):
    
    for i in html.find_all('div', class_='hds-leading-[normal] hds-text-ellipsis hds-space-x-2 hds-block'):
        brand.append(i.find_all('span', class_='hds-text-body-medium hds-text-gray-tertiary')[0].text)
        title.append(i.find_all('span', class_='hds-text-body-medium hds-text-gray-primary')[0].text)
    
    for i in html.find_all('a', class_='flex items-center'):
        if i.find_all('span', class_= 'hds-text-subtitle-small'):
            discount.append(i.find_all('span', class_= 'hds-text-subtitle-small')[0].text)
        else:
            discount.append('0')  # 할인이 없는 경우 0으로 표기
        if i.find_all('span', class_ = 'hds-text-gray-primary hds-text-subtitle-large'):
            price.append(i.find_all('span', class_ = 'hds-text-gray-primary hds-text-subtitle-large')[0].text)
        else:
            price.append(i.find_all('span', class_ = 'hds-text-body-large hds-text-gray-secondary')[0].text)
        if i.find_all('span', class_ = 'hds-text-smalltext-large')[1].text == '정가':
            etc.append(i.find_all('span', class_ = 'hds-text-smalltext-large')[2].text)
        else:
            etc.append(i.find_all('span', class_ = 'hds-text-smalltext-large')[1].text[1:])
    
    for i in html.find_all('div', class_='hds-flex hds-items-center hds-space-x-2'):
        star.append(i.text[:4])
        starlen.append(i.text[4:])

    origin_url = 'https://www.hwahae.co.kr/'
    review_box = []
    a_tag = html.find_all('a', class_= 'flex items-center')
    for i in range(len(a_tag)):
        box = []
        inner_url = requests.get(origin_url+a_tag[i]['href'], headers=dic)
        inner_html = BeautifulSoup(inner_url.text)
        good_reviews = inner_html.find('div', class_='grow mr-24 w-1/2').find_all('span',class_='hds-text-caption-large text-gray-primary line-clamp-1')
        good_reviews_len = inner_html.find('div', class_='grow mr-24 w-1/2').find_all('span', class_ = 'hds-text-body-medium text-gray-tertiary')
        for j in range(len(good_reviews)):
            box.append((good_reviews[j].text, int(good_reviews_len[j].text.replace(',', ''))))
        temp_df = pd.DataFrame(box, columns = ['리뷰', '리뷰개수'])
        temp_df['리뷰비율'] = round(temp_df['리뷰개수'] / sum(temp_df['리뷰개수']),2)
        review_box.append(temp_df[['리뷰비율','리뷰']].set_index('리뷰').T.reset_index(drop=True))
    review_df = pd.concat(review_box).fillna(0).reset_index(drop=True)

    
    data = {
    'Brand': brand,
    'Title': title,
    'Discount': discount,
    'Price': price,
    'Etc': etc,
    'Star': star,
    'StarLen': starlen
    }

    df = pd.DataFrame(data)
    
    return df, review_df

In [7]:
age_type = [1372, 1543, 1714, 1885]

age_type_mapping = {
    1372: '10대',
    1543: '20대',
    1714: '30대',
    1885: '40대 이상'
}

total = []
total2 = []
for age in age_type:
    brand = []
    title = []
    discount = []
    price = []
    etc = []
    star = []
    starlen = []
    
    dic = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 14_7)'
    }
    url = requests.get(f'https://www.hwahae.co.kr/rankings?english_name=age&theme_id={age}', headers=dic)
    html = BeautifulSoup(url.text)

    temp, temp2 = make_data(html, brand, title, discount, price, etc, star, starlen)
    temp['age'] = age_type_mapping[age]
    
    total.append(temp)
    total2.append(temp2)
    print(f'{age_type_mapping[age]} 크롤링 완료')

product_info = pd.concat(total).reset_index(drop=True)
review_info = pd.concat(total2).reset_index(drop=True)

10대 크롤링 완료
20대 크롤링 완료
30대 크롤링 완료
40대 이상 크롤링 완료


In [8]:
result = pd.concat([product_info, review_info],axis=1)

# StarLen의 쉼표 제거 후 정수로 변환
result['StarLen'] = result['StarLen'].str.replace(',', '').astype(float)

# 나이대 컬럼 생성
age_col = ['10대', '20대', '30대', '40대 이상']

# 나이대별로 이진 열 변환
for age in age_col:
    result[age] = result['age'].apply(lambda x: 1 if age == x else 0)

# 'Etc' 뒤에 있는 나머지 열들 가져오기
other_columns = result.columns[result.columns.get_loc('Etc') + 1:]

# 그룹화 및 집계
result_grouped = result.groupby(['Brand', 'Title', 'Discount', 'Price', 'Etc']).agg({
    'Star': 'mean',            # Star의 평균
    'StarLen': 'mean',         # StarLen의 평균
    '10대': 'max',             # 나이대는 최대값으로 통합
    '20대': 'max',
    '30대': 'max',
    '40대 이상': 'max',
    **{col: 'max' for col in other_columns}  # 나머지 열들을 max로 집계
}).reset_index()

# Star와 StarLen 열 포맷팅
result_grouped['Star'] = result_grouped['Star'].round(2)
result_grouped['StarLen'] = result_grouped['StarLen'].round(0).astype(int)

In [18]:
result_grouped = result_grouped.fillna(0)

In [19]:
result_grouped.to_csv('age.csv',index=False)